# STEP 6: LangGraph Conditional Edge

이 노트북의 목표는 State의 값에 따라 LangGraph가 **서로 다른 Node 경로를 선택하는 방법**을 이해하는 것입니다.

`재무 데이터 준비 → 조건 확인 → 기본 요약 또는 영업이익률 계산 → 요약`

이번 단계에서는 가짜 재무 데이터를 사용하고, `analysis_type`도 직접 넣습니다. LLM, OpenDART API, Tool, Agent는 사용하지 않습니다.

## 0. 이번 분기와 LLM Router의 차이

현재는 사람이 `analysis_type`을 `basic` 또는 `margin`으로 정합니다. 조건 함수는 그 값을 읽어 다음 Node 이름을 반환합니다.

따라서 이것은 **Conditional Edge의 동작**을 보는 예제이지, 질문을 LLM이 해석하는 Router는 아닙니다. LLM Router는 이후 단계에서만 다룹니다.

## 1. 구성요소 불러오기

`Literal`은 `analysis_type`과 조건 함수의 반환값에 가능한 값을 명확히 적기 위한 타입 표기입니다.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph

## 2. State 정의

처음에는 회사명, 질문, 분석 유형만 있습니다. `prepare_financials` Node가 가짜 매출액과 영업이익을 추가하고, 이후 경로에 따라 `operating_margin` 또는 `summary`가 추가됩니다.

In [ ]:
class FinancialAnalysisState(TypedDict, total=False):
    company_name: str
    question: str
    analysis_type: Literal["basic", "margin"]
    revenue: int
    operating_income: int
    operating_margin: float
    summary: str

## 3. 첫 번째 Node: 가짜 재무 데이터 준비

나중에는 이 Node가 OpenDART 조회 결과를 State에 넣을 수 있습니다. 지금은 Conditional Edge만 보기 위해 고정된 값을 반환합니다.

In [ ]:
def prepare_financials(state: FinancialAnalysisState) -> dict[str, int]:
    return {"revenue": 1_000, "operating_income": 120}

## 4. 조건 함수: 다음 Node 이름 선택

이 함수는 Node가 아닙니다. State를 읽고 다음에 이동할 Node의 이름만 반환합니다. `add_conditional_edges()`가 이 반환값을 실제 Edge와 연결합니다.

In [ ]:
def route_by_analysis_type(
    state: FinancialAnalysisState,
) -> Literal["basic_summary", "calculate_margin"]:
    if state["analysis_type"] == "basic":
        return "basic_summary"
    if state["analysis_type"] == "margin":
        return "calculate_margin"
    raise ValueError("analysis_type은 'basic' 또는 'margin'이어야 합니다.")

## 5. `basic` 경로의 Node

기본 재무 조회 요청이라면 매출액과 영업이익만 요약하고 종료합니다. 영업이익률 계산 Node를 거치지 않습니다.

In [ ]:
def create_basic_summary(state: FinancialAnalysisState) -> dict[str, str]:
    summary = (
        f'{state["company_name"]}의 매출액은 {state["revenue"]}, '
        f'영업이익은 {state["operating_income"]}입니다.'
    )
    return {"summary": summary}

## 6. `margin` 경로의 Node

영업이익률 분석 요청은 계산 Node를 먼저 거친 뒤, 계산된 값을 사용하는 요약 Node로 이동합니다.

In [ ]:
def calculate_operating_margin(state: FinancialAnalysisState) -> dict[str, float]:
    margin = state["operating_income"] / state["revenue"] * 100
    return {"operating_margin": round(margin, 2)}

In [ ]:
def create_margin_summary(state: FinancialAnalysisState) -> dict[str, str]:
    summary = f'{state["company_name"]}의 영업이익률은 {state["operating_margin"]}%입니다.'
    return {"summary": summary}

## 7. Conditional Edge를 포함한 Graph 만들기

흐름은 아래와 같습니다.

`START → prepare_financials → [조건 분기]`

- `basic` → `basic_summary` → `END`
- `margin` → `calculate_margin` → `margin_summary` → `END`

In [ ]:
workflow = StateGraph(FinancialAnalysisState)

workflow.add_node("prepare_financials", prepare_financials)
workflow.add_node("basic_summary", create_basic_summary)
workflow.add_node("calculate_margin", calculate_operating_margin)
workflow.add_node("margin_summary", create_margin_summary)

workflow.add_edge(START, "prepare_financials")
workflow.add_conditional_edges(
    "prepare_financials",
    route_by_analysis_type,
    {
        "basic_summary": "basic_summary",
        "calculate_margin": "calculate_margin",
    },
)
workflow.add_edge("basic_summary", END)
workflow.add_edge("calculate_margin", "margin_summary")
workflow.add_edge("margin_summary", END)

app = workflow.compile()

## 8. Graph 구조 확인

출력된 Mermaid 텍스트에서 `prepare_financials` 다음에 두 경로가 있는지 확인하세요.

In [ ]:
print(app.get_graph().draw_mermaid())

## 9. `basic` 경로 실행

`analysis_type`이 `basic`이면 `calculate_margin` 없이 `basic_summary`로 이동합니다.

In [ ]:
basic_input: FinancialAnalysisState = {
    "company_name": "가상기업",
    "question": "매출액과 영업이익을 알려줘.",
    "analysis_type": "basic",
}

basic_result = app.invoke(basic_input)
basic_result

## 10. `margin` 경로 실행

`analysis_type`이 `margin`이면 `calculate_margin`을 거쳐 `margin_summary`로 이동합니다.

In [ ]:
margin_input: FinancialAnalysisState = {
    "company_name": "가상기업",
    "question": "영업이익률을 알려줘.",
    "analysis_type": "margin",
}

margin_result = app.invoke(margin_input)
margin_result

## 11. 두 경로가 달랐는지 확인

이 확인은 정량 평가가 아니라 Conditional Edge가 의도한 경로를 선택했는지 보는 최소 학습용 점검입니다.

In [ ]:
assert "operating_margin" not in basic_result
assert margin_result["operating_margin"] == 12.0

print("analysis_type에 따라 서로 다른 경로가 실행되었습니다.")

## 이번 단계에서 기억할 점

- **Conditional Edge**는 State를 읽고 다음 Node를 선택합니다.
- 조건 함수는 보통 Node처럼 State를 갱신하지 않고, 경로 이름을 반환합니다.
- 현재 `analysis_type`은 사람이 직접 넣은 값입니다. 이것을 LLM이 구조화된 값으로 만들게 하는 것은 별도의 다음 단계입니다.
- 조건 분기는 여러 Tool, 검색, 계산 중 무엇을 실행할지 선택해야 하는 최종 기업 분석 workflow의 기반입니다.